# Lesson 2 — MCP Tools (campus_tools_mcp)

**Audience:** University faculty (Malaysia)  
**Goal:** Build a small **MCP tool server** (stdio-ready) exposing campus-like utilities, then **test tools directly** (no LLM yet).

---

## What you will build

1. A **FastMCP** server named `campus_tools_mcp` with typed, schema-friendly outputs  
2. A tiny **fictional campus dataset** (JSON)  
3. Direct tool testing using an MCP client session (in-memory for reliability)  
4. A deliberate **failure demo** (invalid / missing arguments)

---

## Why this lesson matters

- **MCP = Tools interoperability** (standard way to expose functions/data to agents)  
- You will reuse these tools in Lesson 3 where an LLM decides *which tool to call*.

**Key idea:** tools should return **structured outputs** (lists/dicts with fixed keys). This makes tool-using agents safer and more reliable.


## 1. Setup

### 1.1 Install dependencies (run once)

You need the MCP Python SDK.

> If you already installed MCP in Lesson 1/your environment, you can skip this cell.


In [1]:
# If needed, uncomment and run:
!pip -q install "mcp" 

In [2]:
import mcp
print("✅ MCP installed:", getattr(mcp, "__version__", "unknown"))

✅ MCP installed: unknown


## 2. Create a fictional campus dataset

We will store simple records for:

- Staff directory (name, department, email, office)
- Office hours (by staff name)
- Timetable slots (course code, week, day, time, venue)
- Room directory (building, room, capacity, facilities)
- Contacts (helpdesk-like directory)

This is **fictional**, but feels real for teaching.


In [3]:
%%writefile campus_data.json
{
  "staff": [
    {"name": "Dr Amina Rahman", "department": "Computer Science", "email": "amina.rahman@uni.example", "office": "CS-3.21"},
    {"name": "Prof Lim Wei Jian", "department": "Information Systems", "email": "lim.weijian@uni.example", "office": "IS-2.14"},
    {"name": "Dr Nur Syafiqah", "department": "Computer Science", "email": "nur.syafiqah@uni.example", "office": "CS-3.08"},
    {"name": "Ms Tan Mei Ling", "department": "Academic Office", "email": "tan.meiling@uni.example", "office": "AO-1.03"}
  ],
  "office_hours": [
    {"staff_name": "Dr Amina Rahman", "day": "Tue", "time": "14:00-16:00", "mode": "In-person", "location": "CS-3.21"},
    {"staff_name": "Dr Amina Rahman", "day": "Thu", "time": "10:00-11:00", "mode": "Online", "location": "MS Teams"},
    {"staff_name": "Prof Lim Wei Jian", "day": "Wed", "time": "09:00-11:00", "mode": "In-person", "location": "IS-2.14"}
  ],
  "timetable": [
    {"course_code": "SENG3200", "week": 3, "day": "Mon", "time": "10:00-12:00", "venue": "LT-1"},
    {"course_code": "SENG3200", "week": 3, "day": "Thu", "time": "14:00-16:00", "venue": "CS-Lab-2"},
    {"course_code": "DSAI2101", "week": 3, "day": "Fri", "time": "09:00-11:00", "venue": "LT-2"}
  ],
  "rooms": [
    {"building": "LT", "room": "LT-1", "capacity": 180, "facilities": ["Projector", "PA System"]},
    {"building": "LT", "room": "LT-2", "capacity": 120, "facilities": ["Projector"]},
    {"building": "CS", "room": "CS-Lab-2", "capacity": 35, "facilities": ["PCs", "Whiteboard", "HDMI"]},
    {"building": "LIB", "room": "LIB-Meeting-1", "capacity": 12, "facilities": ["TV", "Whiteboard"]}
  ],
  "contacts": [
    {"contact_type": "it_helpdesk", "name": "IT Helpdesk", "email": "helpdesk@uni.example", "phone": "+60-3-5555-1000"},
    {"contact_type": "library", "name": "Library Counter", "email": "library@uni.example", "phone": "+60-3-5555-2000"},
    {"contact_type": "academic_office", "name": "Academic Office", "email": "academics@uni.example", "phone": "+60-3-5555-3000"}
  ]
}


Overwriting campus_data.json


In [4]:
import json
from pathlib import Path

DATA_PATH = Path("campus_data.json")
data = json.loads(DATA_PATH.read_text())

print("Staff:", len(data["staff"]))
print("Office hours:", len(data["office_hours"]))
print("Timetable slots:", len(data["timetable"]))
print("Rooms:", len(data["rooms"]))
print("Contacts:", len(data["contacts"]))

Staff: 4
Office hours: 3
Timetable slots: 3
Rooms: 4
Contacts: 3


## 3. Build the MCP server (FastMCP, stdio-ready)

We will implement the server in `campus_tools_mcp.py`.

### Design principles
- Tools return **structured outputs** (dict/list of dict) with fixed keys
- Minimal input validation
- Clear errors (for teaching)

> In Lesson 3, the LLM will call these tools. Structured outputs make that reliable.


In [5]:
%%writefile campus_tools_mcp.py
from __future__ import annotations

import json
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Optional

from mcp.server.fastmcp import FastMCP
from dataclasses import dataclass
from typing import Optional


# -----------------------------------------------------------------------------
# Server
# -----------------------------------------------------------------------------
# json_response=True enables structuredContent in tool outputs (useful for clients)
mcp = FastMCP("campus_tools_mcp", json_response=True)

DATA_PATH = Path(__file__).with_name("campus_data.json")
DATA = json.loads(DATA_PATH.read_text())


# -----------------------------------------------------------------------------
# Typed records (schema-friendly)
# -----------------------------------------------------------------------------

@dataclass
class ErrorResult:
    error: str

@dataclass
class OfficeHoursResponse:
    staff_name: str
    office_hours: list[OfficeHours]


@dataclass
class StaffRecord:
    name: str
    department: str
    email: str
    office: str


@dataclass
class OfficeHours:
    staff_name: str
    day: str
    time: str
    mode: str
    location: str


@dataclass
class TimetableSlot:
    course_code: str
    week: int
    day: str
    time: str
    venue: str


@dataclass
class RoomInfo:
    building: str
    room: str
    capacity: int
    facilities: list[str]


@dataclass
class Contact:
    contact_type: str
    name: str
    email: str
    phone: str


def _norm(s: str) -> str:
    return s.strip().lower()


@mcp.tool()
def find_staff(query: str, department: Optional[str] = None) -> list[dict]:
    """Find staff members by name fragment, optionally filtered by department."""
    q = _norm(query)
    dept = _norm(department) if department else None

    results: list[StaffRecord] = []
    for s in DATA["staff"]:
        if q in _norm(s["name"]) and (dept is None or dept == _norm(s["department"])):
            results.append(StaffRecord(**s))

    return [asdict(r) for r in results]

@mcp.tool()
def get_office_hours(staff_name: str) -> list[dict]:
    target = _norm(staff_name)
    matches: list[OfficeHours] = []
    for oh in DATA["office_hours"]:
        if _norm(oh["staff_name"]) == target:
            matches.append(OfficeHours(**oh))

    if not matches:
        return [{"error": f"No office hours found for: {staff_name}"}]

    return [{
        "staff_name": staff_name,
        "office_hours": [asdict(m) for m in matches]
    }]


@mcp.tool()
def find_timetable(course_code: str, week: Optional[int] = None) -> list[dict]:
    """Find timetable slots by course code, optionally filtered by week."""
    cc = _norm(course_code)

    if week is not None and int(week) <= 0:
        return [{"error": "week must be a positive integer"}]

    results: list[TimetableSlot] = []
    for t in DATA["timetable"]:
        if _norm(t["course_code"]) == cc and (week is None or int(t["week"]) == int(week)):
            results.append(TimetableSlot(**t))

    return [asdict(r) for r in results]


@mcp.tool()
def find_room(building: str, room: str) -> list[dict]:
    b = _norm(building)
    r = _norm(room)

    for item in DATA["rooms"]:
        if _norm(item["building"]) == b and _norm(item["room"]) == r:
            return [asdict(RoomInfo(**item))]

    return [{"error": f"Room not found: building={building}, room={room}"}]

@mcp.tool()
def list_contacts(contact_type: str) -> list[dict]:
    """List contacts by type."""
    ct = _norm(contact_type)
    results: list[Contact] = []
    for c in DATA["contacts"]:
        if _norm(c["contact_type"]) == ct:
            results.append(Contact(**c))

    return [asdict(r) for r in results]


if __name__ == "__main__":
    # Default transport is stdio for FastMCP servers.
    mcp.run()


Overwriting campus_tools_mcp.py


## 4. Test tools directly (no LLM)

### Option A (recommended for class): In-memory client session

This avoids terminal/process complexity and is highly reliable in workshops.

We will:
- list tools (capability discovery)
- call each tool and inspect `structuredContent`


In [6]:
import asyncio
from mcp.shared.memory import create_connected_server_and_client_session
from mcp.types import CallToolResult

import importlib
import campus_tools_mcp
importlib.reload(campus_tools_mcp) # imports the `mcp` app we wrote

async def list_tools_and_call_examples():
    async with create_connected_server_and_client_session(
        campus_tools_mcp.mcp, raise_exceptions=True
    ) as session:
        tools = await session.list_tools()
        tool_names = [t.name for t in tools.tools]
        print("✅ Tools discovered:", tool_names)

        r1: CallToolResult = await session.call_tool("find_staff", {"query": "Amina"})
        print("\nfind_staff('Amina') structuredContent:")
        print(r1.structuredContent)

        r2: CallToolResult = await session.call_tool("get_office_hours", {"staff_name": "Dr Amina Rahman"})
        print("\nget_office_hours('Dr Amina Rahman') structuredContent:")
        print(r2.structuredContent)

        r3: CallToolResult = await session.call_tool("find_timetable", {"course_code": "SENG3200", "week": 3})
        print("\nfind_timetable('SENG3200', week=3) structuredContent:")
        print(r3.structuredContent)

        r4: CallToolResult = await session.call_tool("find_room", {"building": "CS", "room": "CS-Lab-2"})
        print("\nfind_room('CS', 'CS-Lab-2') structuredContent:")
        print(r4.structuredContent)

        r5: CallToolResult = await session.call_tool("list_contacts", {"contact_type": "it_helpdesk"})
        print("\nlist_contacts('it_helpdesk') structuredContent:")
        print(r5.structuredContent)


await list_tools_and_call_examples()



[03/10/26 17:55:44] INFO     Processing request of type ListToolsRequest                              ]8;id=973885;file:///home/robomy/miniconda3/envs/jupyter_env/lib/python3.11/site-packages/mcp/server/lowlevel/server.py\server.py]8;;\:]8;id=364886;file:///home/robomy/miniconda3/envs/jupyter_env/lib/python3.11/site-packages/mcp/server/lowlevel/server.py#709\709]8;;\

✅ Tools discovered: ['find_staff', 'get_office_hours', 'find_timetable', 'find_room', 'list_contacts']


                    INFO     Processing request of type CallToolRequest                               ]8;id=773048;file:///home/robomy/miniconda3/envs/jupyter_env/lib/python3.11/site-packages/mcp/server/lowlevel/server.py\server.py]8;;\:]8;id=41746;file:///home/robomy/miniconda3/envs/jupyter_env/lib/python3.11/site-packages/mcp/server/lowlevel/server.py#709\709]8;;\


find_staff('Amina') structuredContent:
{'result': [{'name': 'Dr Amina Rahman', 'department': 'Computer Science', 'email': 'amina.rahman@uni.example', 'office': 'CS-3.21'}]}


                    INFO     Processing request of type CallToolRequest                               ]8;id=868813;file:///home/robomy/miniconda3/envs/jupyter_env/lib/python3.11/site-packages/mcp/server/lowlevel/server.py\server.py]8;;\:]8;id=749932;file:///home/robomy/miniconda3/envs/jupyter_env/lib/python3.11/site-packages/mcp/server/lowlevel/server.py#709\709]8;;\

...


find_timetable('SENG3200', week=3) structuredContent:
{'result': [{'course_code': 'SENG3200', 'week': 3, 'day': 'Mon', 'time': '10:00-12:00', 'venue': 'LT-1'}, {'course_code': 'SENG3200', 'week': 3, 'day': 'Thu', 'time': '14:00-16:00', 'venue': 'CS-Lab-2'}]}


                    INFO     Processing request of type CallToolRequest                               ]8;id=264313;file:///home/robomy/miniconda3/envs/jupyter_env/lib/python3.11/site-packages/mcp/server/lowlevel/server.py\server.py]8;;\:]8;id=12158;file:///home/robomy/miniconda3/envs/jupyter_env/lib/python3.11/site-packages/mcp/server/lowlevel/server.py#709\709]8;;\


find_room('CS', 'CS-Lab-2') structuredContent:
{'result': [{'building': 'CS', 'room': 'CS-Lab-2', 'capacity': 35, 'facilities': ['PCs', 'Whiteboard', 'HDMI']}]}


                    INFO     Processing request of type CallToolRequest                               ]8;id=546108;file:///home/robomy/miniconda3/envs/jupyter_env/lib/python3.11/site-packages/mcp/server/lowlevel/server.py\server.py]8;;\:]8;id=415753;file:///home/robomy/miniconda3/envs/jupyter_env/lib/python3.11/site-packages/mcp/server/lowlevel/server.py#709\709]8;;\


list_contacts('it_helpdesk') structuredContent:
{'result': [{'contact_type': 'it_helpdesk', 'name': 'IT Helpdesk', 'email': 'helpdesk@uni.example', 'phone': '+60-3-5555-1000'}]}


## 5. Failure demo (important for teaching)

We show two failure types:

1) **Valid call, but no matching data** → tool returns structured `{ "error": ... }`  
2) **Invalid call shape** (missing required parameter) → client raises an exception


In [7]:
import asyncio
from mcp.shared.memory import create_connected_server_and_client_session

import campus_tools_mcp

async def failure_demo():
    async with create_connected_server_and_client_session(
        campus_tools_mcp.mcp, raise_exceptions=False
    ) as session:
        # A) Valid call but no data -> structured error response
        r = await session.call_tool("get_office_hours", {"staff_name": "Dr Ghost Lecturer"})
        print("A) Valid call, no data -> structuredContent:")
        print(r.structuredContent)

        # B) Invalid call (missing required arg) -> exception
        try:
            await session.call_tool("find_room", {"building": "LT"})
        except Exception as e:
            print("\nB) Invalid call (missing arg) -> exception:")
            print(type(e).__name__, ":", e)

await failure_demo()

[03/10/26 17:56:02] INFO     Processing request of type CallToolRequest                               ]8;id=379516;file:///home/robomy/miniconda3/envs/jupyter_env/lib/python3.11/site-packages/mcp/server/lowlevel/server.py\server.py]8;;\:]8;id=517388;file:///home/robomy/miniconda3/envs/jupyter_env/lib/python3.11/site-packages/mcp/server/lowlevel/server.py#709\709]8;;\

                    INFO     Processing request of type ListToolsRequest                              ]8;id=970295;file:///home/robomy/miniconda3/envs/jupyter_env/lib/python3.11/site-packages/mcp/server/lowlevel/server.py\server.py]8;;\:]8;id=798318;file:///home/robomy/miniconda3/envs/jupyter_env/lib/python3.11/site-packages/mcp/server/lowlevel/server.py#709\709]8;;\

A) Valid call, no data -> structuredContent:
{'result': [{'error': 'No office hours found for: Dr Ghost Lecturer'}]}


                    INFO     Processing request of type CallToolRequest                               ]8;id=552642;file:///home/robomy/miniconda3/envs/jupyter_env/lib/python3.11/site-packages/mcp/server/lowlevel/server.py\server.py]8;;\:]8;id=823730;file:///home/robomy/miniconda3/envs/jupyter_env/lib/python3.11/site-packages/mcp/server/lowlevel/server.py#709\709]8;;\

## 6. Run as a real STDIO server (for Lesson 3)

In Lesson 3, your tool-using agent will spawn this server via stdio.

Run in a terminal:

```bash
python campus_tools_mcp.py
```

Nothing “prints” because MCP uses stdin/stdout messages.

### Optional: MCP Inspector
The MCP Python SDK docs mention using the Inspector to explore servers:

```bash
npx -y @modelcontextprotocol/inspector
```

Then connect using the UI (for stdio servers: command = `python campus_tools_mcp.py`).


## 7. Exercises (for faculty participants)

1) Add a tool:
- `list_departments() -> list[str]`

2) Extend `find_staff(...)`:
- add `email_contains: str | None = None`

3) Tighten validation:
- `find_timetable(course_code, week)` should return an error if `week <= 0` (already included). Try changing the error structure.

> Keep outputs schema-friendly (lists/dicts) so Lesson 3 remains stable.
